# Data Generation

This notebook simulates pp collisions at the STAR experiment and stores them into a ROOT tree. Afterwards, the data are smeared to simulate detector effects.

### ROOT Tree Generation Using PYTHIA8

At first, it is necessary to import ROOT, os, and PYTHIA8.

In [ ]:
import ROOT
import os
ROOT.gSystem.Load("libpythia8") # load PYTHIA8 into ROOT

ROOT.EnableImplicitMT() # enable multi-threading

Then, we call the C++ generateEvents(nEvents, seed, outFileName) function with PYTHIA8 MC generator configured for $\sqrt{s_\mathrm{NN}} = 510 \: \mathrm{GeV}$ pp collisions with the Detroit tune for RHIC. The function stores the generated events into a TTree and saves everything into a .root file.

In [ ]:
# set the seed and number of events to generate
seed = 22
nEvents = 2000000

# define the directory and file path
outDir = f"data/{seed}"
os.makedirs(outDir, exist_ok=True) # creates the directory if it doesn't exist, otherwise does nothing
outFileName = f"{outDir}/events{seed}.root"

# load the generateEvents(nEvents, seed, outFileName) function via the ROOT's gInterpreter
ROOT.gInterpreter.ProcessLine('#include "cpp/generate_events.cpp"')

# generate events
ROOT.generateEvents(nEvents, seed, outFileName)

### Simulating Detector Effects

Now, we have the simulated particle collisions from PYTHIA8 (the "true" data) and would like to obtain a data set similar to the ones we measure at STAR. In order to do so, we must simulate the detector effects of the STAR detector.

We are using RDataFrame for work with the data. We define a new data frame containing 4-vectors (tracks) of $p_\mathrm{T}$, $\eta$, $\phi$, mass using LorentzVector (PtEtaPhiMVector).

In [ ]:
import ROOT # tyto 2 radky smazat nakonec
ROOT.EnableImplicitMT()

df = ROOT.RDataFrame("events", f"data/{seed}/events{seed}.root") # connect RDataFrame to the TTree in events{seed}.root

dfTracks = df.Define("tracks", "ROOT::VecOps::Construct<ROOT::Math::PtEtaPhiMVector> (pT, eta, phi, mass)") # create tracks -- put pT, eta, phi, mass into a LorentzVector

!!! SO FAR, THE RESOLUTION AND ACCEPTANCE FUNCTIONS ARE BASED FOR PIONS -- I NEED TO ADD OTHER PARTICLES IN THE FUTURE !!!

Firstly, we simulate the **acceptance of TPC and TOF**.

In [ ]:
# I wrote a function TPCacceptance(Nch, pT) which randomly selects particles in an event based on an experimentally determined TPCs acceptance function (!!! now it's constant for all pT values, but will be modified to match experiments in the future) and a function TPCandTOFacceptance(Nch, pT, TPCaccepted) which works similarly. We need to tell ROOT to load and compile them.
ROOT.gInterpreter.ProcessLine('#include "cpp/TPCandTOFacceptance.cpp"')

dfAccepted = (
    dfTracks.Define("TPCaccepted", "TPCmask(tracks.size(), pT, eventID)") # define a new column with events accepted (1) and rejected (0) by TPC
            .Define("TPCandTOFaccepted", "TPCandTOFmask(tracks.size(), pT, eventID, TPCaccepted)") # define a new column with events accepted (1) and rejected (0) by both TPC and TOF
            .Define("acceptedTracks", "tracks[TPCandTOFaccepted]") # define a new column to store only accepted tracks
)

Next, we need to smear $p_\mathrm{T}$, $\phi$, and $\eta$ to simulate **imperfect resolution of the detector**.

In [ ]:
ROOT.gInterpreter.ProcessLine('#include "cpp/smearing.cpp"') # load C++ function which smears the accepted tracks

dfSmearedRaw = (
    dfAccepted.Define("smearedTracksRaw", "smearing(acceptedTracks, eventID)")
)

We also apply the midrapidity cut $|\eta| < 1$ and the $p_\mathrm{T} > 0.2 \: \mathrm{GeV}$ cut.

In [ ]:
ROOT.gInterpreter.ProcessLine('#include "cpp/extract_components.cpp"') # load C++ functions which extract components of our tracks

dfSmeared = (
    dfSmearedRaw.Define("pTSmearedRaw", "getpT(smearedTracksRaw)") # extract pT
                .Define("etaSmearedRaw", "getEta(smearedTracksRaw)") # extract eta
                .Define("smearedAccepted", "pTSmearedRaw > 0.2 && abs(etaSmearedRaw) < 1.0") # define a new column containing masks for each event, which assign 1 to particles which passed both the pT and midrapidity cut, 0 otherwise
                .Define("smearedTracks", "smearedTracksRaw[smearedAccepted]") # redefine the smearedTracks column to keep only particles which passed both cuts
                .Define("pTSmeared", "pTSmearedRaw[smearedAccepted]")
                .Define("etaSmeared", "etaSmearedRaw[smearedAccepted]")
                .Define("phiSmeared", "getPhi(smearedTracks)")
)

In the end, we extract each 4-vector component into columns and save these columns into a new tree composed of smeared data.

In [ ]:
# save the smeared tree into a .root file
opts = ROOT.RDF.RSnapshotOptions() # ensure that events_smeared.root file is recreated every time this notebook is ran
opts.fMode = "RECREATE"
dfSmeared.Snapshot("events_smeared", f"data/{seed}/events{seed}_smeared.root", ['eventID', 'pT', 'eta', 'phi', 'mass', 'TPCaccepted', 'TPCandTOFaccepted', 'pTSmeared', 'etaSmeared', 'phiSmeared'], opts)